In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==0.4.6
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
tmin = -.5
tmax=2.0
fmin=0.5
fmax = 16
sfreq = 32

paradigm = MotorImagery(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax,  n_classes=7)
datasets = [
    Weibo2014()
]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda_mi",
    overwrite=True,
    random_state=42,
    n_jobs=5,
    #data_size=dict(
    #    policy='per_class',
    #    value=[10]
    #),
    #n_perms=[2],
)

Choosing from all possible events


In [3]:
from sklearn.pipeline import make_pipeline, Pipeline
from mne.decoding import Scaler
from hoda.hoda import HODA, BTTDA
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from pyriemann.estimation import XdawnCovariances
from pyriemann.tangentspace import TangentSpace
from hoda.classification import ToeplitzLDAWrapper, Vectorize
from sklearn.feature_selection import SelectFwe, SelectKBest, RFECV
from sklearn.model_selection import GridSearchCV


pipelines = dict()


pipelines['sLDA'] = make_pipeline(
        Vectorize(),
        LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['HODA_prune'] = Pipeline([
    ('hoda', HODA(
        max_iter=256,
        tol=1e-6,
        init ='svd',
        shrinkage='lw',
        toeplitz=None,
        obj='rt',
        solver='lanczos',
        taper=False,
        keep_train_info=False,
        verbose=False,
        prune=True,
    )),
    ('vec', Vectorize()),
    ('select', SelectFwe()),
    ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

In [4]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

Weibo 2014-WithinSession:   0%|    | 0/10 [00:00<?, ?it/s]Trial data de-meaned and concatenated with a buffer to create cont data
/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:180: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Weibo 2014-WithinSession:  10%| | 1/10 [00:50<07:37, 50.84Trial data de-meaned and concatenated with a buffer to create cont data
/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:180: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Weibo 2014-WithinSession:  20%|▏| 2/10 [01:50<07:29, 56.24Trial data de-meaned and concatenated with a buffer to create cont data
/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:180: FutureWarning: The curren

ValueError: Found array with 1 sample(s) (shape=(1, 60)) while a minimum of 2 is required by LinearDiscriminantAnalysis.

In [ ]:
results

In [ ]:
import seaborn as sns
order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()

sns.barplot(
     data=results,
    y="score", x="dataset", hue="pipeline", hue_order=order.index,
)

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'tLDA', 'HODA_prune')
_  = paired_plot(results, 'tLDA', 'HODA_prune')